In [ ]:
import json, os, glob, tqdm
from PIL import Image
import numpy as np
import subprocess
import imageio
import numpy as np

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

def face_segment(segment_part, mask_path):
    face_segment_anno = imageio.v2.imread(mask_path)

    face_segment_anno = np.array(face_segment_anno)
    bg = (face_segment_anno == 0)
    skin = (face_segment_anno == 1)
    l_brow = (face_segment_anno == 2)
    r_brow = (face_segment_anno == 3)
    l_eye = (face_segment_anno == 4)
    r_eye = (face_segment_anno == 5)
    eye_g = (face_segment_anno == 6)
    l_ear = (face_segment_anno == 7)
    r_ear = (face_segment_anno == 8)
    ear_r = (face_segment_anno == 9)
    nose = (face_segment_anno == 10)
    mouth = (face_segment_anno == 11)
    u_lip = (face_segment_anno == 12)
    l_lip = (face_segment_anno == 13)
    neck = (face_segment_anno == 14)
    neck_l = (face_segment_anno == 15)
    cloth = (face_segment_anno == 16)
    hair = (face_segment_anno == 17)
    hat = (face_segment_anno == 18)
    face = np.logical_or.reduce((skin, l_brow, r_brow, l_eye, r_eye, eye_g, l_ear, r_ear, ear_r, nose, mouth, u_lip, l_lip))

    if segment_part == 'faceseg_bg':
        seg_m = bg
    elif segment_part == 'faceseg_fg':
        seg_m = ~bg
    else: raise NotImplementedError(f"Segment part: {segment_part} is not found!")
    
    out = seg_m
    return out
            
def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

def blending_mask(img_path, mask_path, hdr_from_bg_path):
    if isinstance(mask_path, np.ndarray):
        mask = mask_path / 255.0
    else:
        mask = imageio.v2.imread(mask_path) / 255.0 #[256, 256]
        
    if isinstance(img_path, np.ndarray):
        img = img_path / 255.0
    else:
        img = imageio.v2.imread(img_path).astype(np.float32) / 255.0
        
    blurred_mask = cv2.GaussianBlur(mask, (3, 3), 0)
    # Apply erosion
    kernel = np.ones((3,3),np.uint8)
    eroded_mask = cv2.erode(blurred_mask, kernel, iterations = 1)
    mask = eroded_mask [..., np.newaxis]
    
    bg = imageio.v2.imread(hdr_from_bg_path).astype(np.float32) / 255.0
    # alpha blending
    out = img * mask + bg * (1 - mask)
    out = (out * 255).astype(np.uint8)
    return out